# RAG Evaluation Pipeline — Local Models (RAGAS 0.4.x)

DenizBank internal Qwen 3.1 LLM + Qwen3-Embedding endpoints.  
Fixes `embed_query` compatibility and includes all meaningful RAGAS metrics.

## 1 — Install

In [ ]:
!pip install ragas openai httpx pandas numpy --quiet

## 2 — Endpoint Configuration

In [ ]:
# ── LLM ──
LLM_BASE_URL = "https://api-qwen-31-infer-tmp-automation-test-3.apps.datascience.prod2.deniz.denizbank.com/v1"
LLM_MODEL = "default"

# ── Embedding ──
EMBEDDING_URL = "http://YOUR_EMBEDDING_HOST:PORT/v1/embeddings"  # update
EMBEDDING_MODEL = "qwen3-embedding"
EMBEDDING_API_KEY = "your-key"

## 3 — LLM Setup

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory

llm_client = AsyncOpenAI(base_url=LLM_BASE_URL, api_key="not-needed")
evaluator_llm = llm_factory(LLM_MODEL, provider="openai", client=llm_client)
print(f"\u2705 LLM: {LLM_MODEL}")

## 4 — Embedding Setup (Fixed)

`embedding_factory` returns RAGAS v0.4's `OpenAIEmbeddings` which has `embed_text` / `aembed_text`.  
But `ResponseRelevancy` and `SemanticSimilarity` internally still call `embed_query` / `embed_documents` (Langchain-style).  
This causes `AttributeError: 'OpenAIEmbeddings' object has no attribute 'embed_query'`.

**Fix:** Custom wrapper that implements **both** interfaces — matches your `get_embeddings_batch` exactly.

In [ ]:
import httpx
import numpy as np
from ragas.embeddings.base import BaseRagasEmbeddings


class DenizBankEmbedding(BaseRagasEmbeddings):
    """
    Embedding wrapper compatible with BOTH:
    - RAGAS v0.4 interface (embed_text / aembed_text)
    - Legacy Langchain interface (embed_query / embed_documents)

    Matches your get_embeddings_batch contract exactly.
    """

    def __init__(self, url: str, model: str, api_key: str = "not-needed"):
        self.url = url
        self.model = model
        self.api_key = api_key
        self._client = httpx.Client(timeout=120.0)
        self._async_client = httpx.AsyncClient(timeout=120.0)

    def _headers(self):
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def _embed_batch(self, texts: list[str]) -> list[list[float]]:
        """Core sync batch embed — mirrors your get_embeddings_batch."""
        resp = self._client.post(
            self.url,
            json={"model": self.model, "input": texts},
            headers=self._headers(),
        )
        resp.raise_for_status()
        data = resp.json()["data"]
        return [item["embedding"] for item in sorted(data, key=lambda x: x["index"])]

    async def _aembed_batch(self, texts: list[str]) -> list[list[float]]:
        """Core async batch embed."""
        resp = await self._async_client.post(
            self.url,
            json={"model": self.model, "input": texts},
            headers=self._headers(),
        )
        resp.raise_for_status()
        data = resp.json()["data"]
        return [item["embedding"] for item in sorted(data, key=lambda x: x["index"])]

    # ── Langchain interface (needed by ResponseRelevancy, SemanticSimilarity) ──

    def embed_query(self, text: str) -> list[float]:
        return self._embed_batch([text])[0]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed_batch(texts)

    # ── RAGAS v0.4 interface ──

    def embed_text(self, text: str, **kwargs) -> list[float]:
        return self._embed_batch([text])[0]

    async def aembed_text(self, text: str, **kwargs) -> list[float]:
        result = await self._aembed_batch([text])
        return result[0]

    def embed_texts(self, texts: list[str], **kwargs) -> list[list[float]]:
        return self._embed_batch(texts)

    async def aembed_texts(self, texts: list[str], **kwargs) -> list[list[float]]:
        return await self._aembed_batch(texts)

    # ── Required by some RAGAS internals ──

    def set_run_config(self, run_config):
        pass


evaluator_embeddings = DenizBankEmbedding(
    url=EMBEDDING_URL,
    model=EMBEDDING_MODEL,
    api_key=EMBEDDING_API_KEY,
)
print(f"\u2705 Embeddings: {EMBEDDING_MODEL}")

## 5 — Smoke Test

In [ ]:
import requests, json

# Test LLM
print("Testing LLM...")
r1 = requests.post(
    f"{LLM_BASE_URL}/chat/completions",
    json={
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": "Merhaba, test."}],
        "chat_template_kwargs": {"enable_thinking": False},
        "temperature": 0.1,
    },
)
print(f"  Status: {r1.status_code}")
if r1.ok:
    print(f"  Response: {json.loads(r1.text)['choices'][0]['message']['content'][:100]}")

# Test Embedding — both interfaces
print("\nTesting Embedding (embed_query)...")
vec = evaluator_embeddings.embed_query("test")
print(f"  embed_query OK — dim={len(vec)}")

print("\nTesting Embedding (embed_text)...")
vec2 = evaluator_embeddings.embed_text("test")
print(f"  embed_text OK — dim={len(vec2)}")

print("\n\u2705 All endpoints ready")

## 6 — Build Evaluation Dataset

In [ ]:
from ragas import SingleTurnSample, EvaluationDataset

samples = [
    SingleTurnSample(
        user_input="Basel III kapsam\u0131nda sermaye yeterlilik oranlar\u0131 nelerdir?",
        response=(
            "Basel III kapsam\u0131nda bankalar minimum %4.5 CET1, "
            "%6 Tier 1 ve %8 toplam sermaye oran\u0131 tutmak zorundad\u0131r."
        ),
        retrieved_contexts=[
            "Basel III'e g\u00f6re asgari \u00c7ekirdek Sermaye (CET1) oran\u0131 %4.5'tir. "
            "Asgari Tier 1 sermaye oran\u0131 %6, toplam sermaye oran\u0131 ise %8'dir.",
            "Basel III ayr\u0131ca %2.5'lik sermaye koruma tamponu getirmi\u015ftir.",
        ],
        reference=(
            "Basel III minimum CET1 %4.5, Tier 1 %6, toplam sermaye %8 "
            "ve %2.5 sermaye koruma tamponu \u00f6ng\u00f6r\u00fcr."
        ),
    ),
    SingleTurnSample(
        user_input="FAISS yakla\u015f\u0131k en yak\u0131n kom\u015fu aramas\u0131n\u0131 nas\u0131l yapar?",
        response=(
            "FAISS, b\u00fcy\u00fck \u00f6l\u00e7ekli vekt\u00f6r veri setlerinde h\u0131zl\u0131 ANN aramas\u0131 i\u00e7in "
            "ters dosya indeksleri (IVF) ve \u00e7arp\u0131m nicelemeyi (PQ) birle\u015ftirir."
        ),
        retrieved_contexts=[
            "FAISS, verimli benzerlik aramas\u0131 i\u00e7in IVF ve PQ uygular.",
            "Milyar \u00f6l\u00e7ekli veri setlerinde FAISS, alt-do\u011frusal arama s\u00fcresi sa\u011flar.",
        ],
        reference="FAISS, \u00f6l\u00e7ekte ANN aramas\u0131 i\u00e7in IVF ve \u00e7arp\u0131m niceleme kullan\u0131r.",
    ),
    SingleTurnSample(
        user_input="LightGBM ve XGBoost aras\u0131ndaki fark nedir?",
        response=(
            "LightGBM histogram tabanl\u0131 b\u00f6lme ve yaprak bazl\u0131 b\u00fcy\u00fcme kullan\u0131r, daha h\u0131zl\u0131d\u0131r. "
            "XGBoost varsay\u0131lan olarak seviye bazl\u0131 b\u00fcy\u00fcme ve kesin a\u00e7g\u00f6zl\u00fc b\u00f6lme kullan\u0131r."
        ),
        retrieved_contexts=[
            "LightGBM histogram tabanl\u0131 karar a\u011fac\u0131 \u00f6\u011frenimi kullan\u0131r ve yaprak bazl\u0131 b\u00fcy\u00fcr.",
            "XGBoost varsay\u0131lan olarak kesin a\u00e7g\u00f6zl\u00fc algoritma kullan\u0131r ve seviye bazl\u0131 b\u00fcy\u00fcr. "
            "tree_method='hist' ile histogram tabanl\u0131 y\u00f6ntemleri de destekler.",
        ],
        reference=(
            "LightGBM histogram tabanl\u0131 yaprak bazl\u0131 b\u00fcy\u00fcme ile daha h\u0131zl\u0131d\u0131r; "
            "XGBoost varsay\u0131lan olarak seviye bazl\u0131 kesin b\u00f6lme yapar."
        ),
    ),
]

eval_dataset = EvaluationDataset(samples=samples)
print(f"\u2705 Dataset: {len(samples)} samples")

### Load from CSV

In [ ]:
# import json
# import pandas as pd
#
# df = pd.read_csv("rag_outputs.csv")
# df["retrieved_contexts"] = df["retrieved_contexts"].apply(json.loads)
# eval_dataset = EvaluationDataset(
#     samples=[SingleTurnSample(**row.to_dict()) for _, row in df.iterrows()]
# )

## 7 — All Meaningful Metrics

### Metric Reference

| # | Metric | Category | What it measures | Needs `reference`? | Uses embeddings? |
|---|--------|----------|------------------|--------------------|------------------|
| 1 | **Faithfulness** | Generator | Response grounded in contexts (no hallucination) | No | No |
| 2 | **ResponseRelevancy** | Generator | Response relevant to the question | No | **Yes** |
| 3 | **FactualCorrectness** | Generator | Response matches ground truth | **Yes** | No |
| 4 | **SemanticSimilarity** | Generator | Embedding similarity: response vs reference | **Yes** | **Yes** |
| 5 | **LLMContextPrecisionWithoutReference** | Retriever | Are retrieved chunks relevant to query? | No | No |
| 6 | **LLMContextRecall** | Retriever | Do contexts cover the reference answer? | **Yes** | No |
| 7 | **ContextEntityRecall** | Retriever | Entity-level recall from reference | **Yes** | No |
| 8 | **NoiseSensitivity** | Retriever | Does irrelevant context cause wrong claims? | **Yes** | No |
| 9 | **ResponseGroundedness** | Generator | Claims supported by context (stricter faithfulness) | No | No |
| 10 | **BleuScore** | Non-LLM | BLEU between response and reference | **Yes** | No |
| 11 | **RougeScore** | Non-LLM | ROUGE-L between response and reference | **Yes** | No |

> **Note on `ResponseRelevancy`:** Your vLLM doesn't support `n>1` completions, so RAGAS gets 1 synthetic question instead of 3.
> This makes the metric noisier but still usable. The `embed_query` fix above should resolve the NaN issue.

In [ ]:
from ragas.metrics import (
    # ── Generator metrics (LLM-based) ──
    Faithfulness,
    ResponseRelevancy,
    FactualCorrectness,
    SemanticSimilarity,
    ResponseGroundedness,

    # ── Retriever metrics (LLM-based) ──
    LLMContextPrecisionWithoutReference,
    LLMContextRecall,
    ContextEntityRecall,
    NoiseSensitivity,

    # ── Non-LLM metrics ──
    BleuScore,
    RougeScore,
)

### 7a — Reference-Free Metrics (no ground truth needed)
Use these when you don't have reference answers.

In [ ]:
reference_free_metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithoutReference(),
    ResponseGroundedness(),
]

### 7b — Reference-Based Metrics (need ground truth)
Use these when you have `reference` field in your samples.

In [ ]:
reference_based_metrics = [
    FactualCorrectness(),
    SemanticSimilarity(),
    LLMContextRecall(),
    ContextEntityRecall(),
    NoiseSensitivity(),
    BleuScore(),
    RougeScore(),
]

### 7c — All Metrics Combined

In [ ]:
all_metrics = reference_free_metrics + reference_based_metrics

## 8 — Run Evaluation

> Always pass **both** `llm` and `embeddings` to prevent OpenAI fallback.

In [ ]:
from ragas import evaluate

results = evaluate(
    dataset=eval_dataset,
    metrics=all_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print(results)

In [ ]:
import pandas as pd

results_df = results.to_pandas()

# Show all metric columns
metric_cols = [
    "faithfulness",
    "response_relevancy",
    "factual_correctness",
    "semantic_similarity",
    "response_groundedness",
    "llm_context_precision_without_reference",
    "llm_context_recall",
    "context_entity_recall",
    "noise_sensitivity",
    "bleu_score",
    "rouge_score",
]

# Filter to columns that actually exist in results
existing_cols = [c for c in metric_cols if c in results_df.columns]
results_df[["user_input"] + existing_cols]

## 9 — Custom Domain Metrics

### 9a — AspectCritic (Binary Pass/Fail)

In [ ]:
from ragas.metrics import AspectCritic

hallucination_check = AspectCritic(
    name="no_hallucination",
    definition=(
        "Yan\u0131t YALNIZCA al\u0131nan ba\u011flamlarda desteklenen bilgileri i\u00e7erir. "
        "Uydurma say\u0131lar, tarihler veya iddialar yoktur."
    ),
    llm=evaluator_llm,
)

completeness_check = AspectCritic(
    name="completeness",
    definition=(
        "Yan\u0131t, kullan\u0131c\u0131n\u0131n sorusunun t\u00fcm k\u0131s\u0131mlar\u0131n\u0131 tam olarak ele al\u0131r "
        "ve ba\u011flamlardaki \u00f6nemli bilgileri atlamaz."
    ),
    llm=evaluator_llm,
)

conciseness_check = AspectCritic(
    name="conciseness",
    definition=(
        "Yan\u0131t gereksiz tekrar, dolgu c\u00fcmleler veya alakas\u0131z bilgi i\u00e7ermez. "
        "Do\u011frudan ve \u00f6z olarak soruya cevap verir."
    ),
    llm=evaluator_llm,
)

custom_results = evaluate(
    dataset=eval_dataset,
    metrics=[hallucination_check, completeness_check, conciseness_check],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

custom_results.to_pandas()[["user_input", "no_hallucination", "completeness", "conciseness"]]

### 9b — DiscreteMetric (Numeric 1-5 Scale)

In [ ]:
from ragas.metrics import DiscreteMetric

technical_depth = DiscreteMetric(
    name="technical_depth",
    allowed_values=list(range(1, 6)),
    prompt="""Yan\u0131t\u0131n teknik derinli\u011fini puanla.
1 = Y\u00fczeysel, detay yok
2 = Temel, n\u00fcanslar\u0131 eksik
3 = Yeterli
4 = \u0130yi derinlik, do\u011fru teknik detaylar
5 = Uzman d\u00fczeyinde hassasiyet

Kullan\u0131c\u0131 sorusu: {user_input}
Yan\u0131t: {response}

Sadece say\u0131 ile cevap ver (1-5).""",
)

score = await technical_depth.ascore(
    llm=evaluator_llm,
    user_input=samples[0].user_input,
    response=samples[0].response,
)
print(f"Technical depth: {score.value}/5 \u2014 {score.reason}")

## 10 — Batch Pipeline Helper

In [ ]:
import requests, json
from typing import Optional


def run_rag_pipeline(query: str, retriever) -> dict:
    """
    TODO: Replace with your actual pipeline.
    1. Retrieve contexts via retriever (FAISS, etc.)
    2. Call LLM with query + context
    """
    # contexts = retriever.search(query, top_k=5)
    # context_str = "\n".join(contexts)
    #
    # resp = requests.post(
    #     f"{LLM_BASE_URL}/chat/completions",
    #     json={
    #         "model": LLM_MODEL,
    #         "messages": [
    #             {"role": "system", "content": "Sadece verilen metinlerdeki bilgileri kullanarak cevap ver."},
    #             {"role": "user", "content": f"Metin: {context_str}. Soru: {query}. Cevap:"},
    #         ],
    #         "chat_template_kwargs": {"enable_thinking": False},
    #         "temperature": 0.1,
    #     },
    # )
    # response_text = json.loads(resp.text)["choices"][0]["message"]["content"]
    # return {"response": response_text, "retrieved_contexts": contexts}
    raise NotImplementedError("Wire up your retriever + LLM")


def evaluate_pipeline(
    queries: list[str],
    retriever,
    references: Optional[list[str]] = None,
    metrics=None,
) -> pd.DataFrame:
    _samples = []
    for i, q in enumerate(queries):
        out = run_rag_pipeline(q, retriever)
        _samples.append(SingleTurnSample(
            user_input=q,
            response=out["response"],
            retrieved_contexts=out["retrieved_contexts"],
            reference=references[i] if references else None,
        ))

    res = evaluate(
        dataset=EvaluationDataset(samples=_samples),
        metrics=metrics or all_metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    return res.to_pandas()

## 11 — Results Analysis & Export

In [ ]:
def analyze_results(df: pd.DataFrame, cols: list[str], threshold: float = 0.7):
    """Aggregate stats and flag weak samples."""
    valid_cols = [c for c in cols if c in df.columns]
    summary = df[valid_cols].describe().T[["mean", "std", "min", "25%", "50%"]]
    summary.columns = ["mean", "std", "min", "p25", "median"]

    for col in valid_cols:
        df[f"{col}_flag"] = df[col].apply(lambda x: x < threshold if pd.notna(x) else True)

    flag_cols = [f"{c}_flag" for c in valid_cols]
    flagged = df[df[flag_cols].any(axis=1)]
    print(f"\n\u26a0\ufe0f  {len(flagged)}/{len(df)} samples below {threshold}")
    if len(flagged) > 0:
        display(flagged[["user_input"] + valid_cols])

    return summary


summary = analyze_results(results_df, metric_cols)
print("\n\U0001f4ca Metric Summary:")
summary

In [ ]:
# Heatmap view
import pandas as pd

styled = results_df[["user_input"] + existing_cols].style.background_gradient(
    cmap="RdYlGn", subset=existing_cols, vmin=0, vmax=1
).format({c: "{:.3f}" for c in existing_cols}, na_rep="NaN")
styled

In [ ]:
results_df.to_csv("eval_results.csv", index=False)
summary.to_csv("eval_summary.csv")
print("\u2705 Exported")

## Notes & Troubleshooting

**`embed_query` fix (Section 4):**  
Root cause: RAGAS v0.4 `OpenAIEmbeddings` has `embed_text` but NOT `embed_query`.  
Metrics like `ResponseRelevancy` and `SemanticSimilarity` still call `embed_query` internally.  
Our `DenizBankEmbedding` class implements both interfaces, fixing the NaN issue.

**"1 generations instead of 3" warning:**  
Your vLLM doesn't support `n>1` completions per request. RAGAS falls back to 1 generation.  
This makes `ResponseRelevancy` noisier (fewer synthetic questions for cosine similarity).  
Other metrics are unaffected. No fix possible server-side — this is a vLLM limitation.

**Metric selection guidance:**
- **No ground truth?** Use `reference_free_metrics` only (Faithfulness, ResponseRelevancy, ContextPrecision, ResponseGroundedness)
- **Have ground truth?** Use `all_metrics` for full picture
- **Quick sanity check?** Faithfulness + FactualCorrectness are the most informative pair
- **Retriever tuning?** Focus on ContextPrecision + ContextRecall + ContextEntityRecall
- **BleuScore / RougeScore:** Useful as non-LLM baselines but weak on semantic understanding — treat as sanity checks, not primary metrics

**Turkish content:**  
Custom `AspectCritic` definitions are in Turkish for better consistency with Qwen on TR content.  
The built-in RAGAS metrics use English prompts internally — this is fine, Qwen handles mixed-language eval well.

**Concurrency:**  
For large eval sets, control parallelism with `RunConfig`:
```python
from ragas.run_config import RunConfig
results = evaluate(..., run_config=RunConfig(max_workers=4))
```